In [ ]:
#  Mount Drive + Extract Eyes Defy Anemia
from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = '/content/drive/MyDrive/anemia_project'

# Re-extract Eyes Defy Anemia (clean zip)
if not os.path.exists('/content/fresh_dataset/dataset anemia/Italy'):
    print("Extracting Eyes Defy Anemia...")
    !apt-get install -y p7zip-full -q
    !7z x /content/drive/MyDrive/anemia_project/dataset_clean.zip -o/content/fresh_dataset -y
    print("Done")
else:
    print("Eyes Defy Anemia already extracted!")

italy_ok = os.path.exists('/content/fresh_dataset/dataset anemia/Italy')
india_ok = os.path.exists('/content/fresh_dataset/dataset anemia/India')
print(f"Italy : {italy_ok}")
print(f"India : {india_ok}")

Mounted at /content/drive
Extracting Eyes Defy Anemia...
Reading package lists...
Building dependency tree...
Reading state information...
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs AMD EPYC 7B12 (830F10),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/anemia_project/                                                1 file, 666269625 bytes (636 MiB)

Extracting archive: /content/drive/MyDrive/anemia_project/dataset_clean.zip
--
Path = /content/drive/MyDrive/anemia_project/dataset_clean.zip
Type = zip
Physical Size = 666269625

  0%      2% 24 - dataset anemia/India/14/20200204_154215_palpebral.png

In [ ]:
#  Download CP-AnemiC from Kaggle
import os, json

os.environ['KAGGLE_USERNAME'] = ""  # paste yours
os.environ['KAGGLE_KEY']      = ""   # paste yours

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'],
               "key"     : os.environ['KAGGLE_KEY']}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install kaggle -q

print("Downloading CP-AnemiC dataset from Kaggle...")
!kaggle datasets download -d kritagyadev/cp-anemic-dataset-from-ghana -p /content/cpanemic

print()
print("Files downloaded:")
for f in os.listdir('/content/cpanemic'):
    size = os.path.getsize(f'/content/cpanemic/{f}') / (1024*1024)
    print(f"  {f}  ({size:.1f} MB)")

Dataset URL: https://www.kaggle.com/datasets/kritagyadev/cp-anemic-dataset-from-ghana
License(s): unknown
100% 7.92M/7.92M [00:00<00:00, 69.4MB/s]


Files downloaded:
  cp-anemic-dataset-from-ghana.zip  (7.9 MB)


In [ ]:
#  Extract CP-AnemiC
import zipfile, os

ZIP_CP   = '/content/cpanemic/cp-anemic-dataset-from-ghana.zip'
DEST_CP  = '/content/cpanemic/extracted'

os.makedirs(DEST_CP, exist_ok=True)

print("Extracting CP-AnemiC...")
with zipfile.ZipFile(ZIP_CP, 'r') as z:
    z.extractall(DEST_CP)

print("Done!")
print()
print("Contents:")
for root, dirs, files in os.walk(DEST_CP):
    depth = root.replace(DEST_CP, '').count(os.sep)
    if depth <= 3:
        indent = '  ' * depth
        folder = os.path.basename(root)
        n_files = len(files)
        print(f"{indent}{folder}/  ({n_files} files)")

Extracting CP-AnemiC...
Done!

Contents:
extracted/  (1 files)
  Non-anemic/  (286 files)
  Anemic/  (424 files)


In [ ]:
#  Build combined label dictionary
# Eyes Defy Anemia + CP-AnemiC combined
import os, json, pandas as pd

DRIVE_PATH   = '/content/drive/MyDrive/anemia_project'
ITALY_PATH   = '/content/fresh_dataset/dataset anemia/Italy'
INDIA_PATH   = '/content/fresh_dataset/dataset anemia/India'
CP_ANEMIC    = '/content/cpanemic/extracted/Anemic'
CP_NORMAL    = '/content/cpanemic/extracted/Non-anemic'

#  PART 1: Eyes Defy Anemia (load existing labels)
with open(os.path.join(DRIVE_PATH, 'image_labels.json'), 'r') as f:
    eyes_defy_dict = json.load(f)

eyes_anemic = sum(1 for v in eyes_defy_dict.values() if v == 1)
eyes_normal = sum(1 for v in eyes_defy_dict.values() if v == 0)
print(f"Eyes Defy Anemia : {len(eyes_defy_dict)} images")
print(f"  Anemic (1) : {eyes_anemic}")
print(f"  Normal (0) : {eyes_normal}")
print()

#  PART 2: CP-AnemiC
cp_dict = {}

# Anemic folder → label 1
for f in os.listdir(CP_ANEMIC):
    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
        cp_dict[os.path.join(CP_ANEMIC, f)] = 1

# Non-anemic folder → label 0
for f in os.listdir(CP_NORMAL):
    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
        cp_dict[os.path.join(CP_NORMAL, f)] = 0

cp_anemic = sum(1 for v in cp_dict.values() if v == 1)
cp_normal = sum(1 for v in cp_dict.values() if v == 0)
print(f"CP-AnemiC Ghana  : {len(cp_dict)} images")
print(f"  Anemic (1) : {cp_anemic}")
print(f"  Normal (0) : {cp_normal}")
print()

#  PART 3: Combine both
combined_dict = {}
combined_dict.update(eyes_defy_dict)
combined_dict.update(cp_dict)

total_anemic = sum(1 for v in combined_dict.values() if v == 1)
total_normal = sum(1 for v in combined_dict.values() if v == 0)
total        = len(combined_dict)

print(f"COMBINED DATASET")
print(f"  Total images : {total}")
print(f"  Anemic (1)   : {total_anemic}  ({100*total_anemic/total:.1f}%)")
print(f"  Normal (0)   : {total_normal}  ({100*total_normal/total:.1f}%)")
print()

# Save combined dictionary
combined_path = os.path.join(DRIVE_PATH, 'combined_labels.json')
with open(combined_path, 'w') as f:
    json.dump(combined_dict, f, indent=2)
print(f"Saved: combined_labels.json ({total} entries)")

Eyes Defy Anemia : 200 images
  Anemic (1) : 75
  Normal (0) : 125

CP-AnemiC Ghana  : 710 images
  Anemic (1) : 424
  Normal (0) : 286

COMBINED DATASET
  Total images : 910
  Anemic (1)   : 499  (54.8%)
  Normal (0)   : 411  (45.2%)

Saved: combined_labels.json (910 entries)


In [ ]:
#  Preprocess all 910 images + split
import cv2, numpy as np, os, json, random, shutil
from sklearn.model_selection import train_test_split

DRIVE_PATH = '/content/drive/MyDrive/anemia_project'

# Load combined dictionary
with open(os.path.join(DRIVE_PATH, 'combined_labels.json'), 'r') as f:
    combined_dict = json.load(f)

all_paths  = list(combined_dict.keys())
all_labels = list(combined_dict.values())

# CLAHE preprocessing function
def preprocess(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.resize(img, (224, 224))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    img = cv2.merge([l, a, b])
    img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)
    return img.astype(np.float32) / 255.0

# Augmentation function
def augment(image):
    img = (image * 255).astype(np.uint8)
    if random.random() > 0.5:
        img = cv2.flip(img, 1)
    angle = random.uniform(-20, 20)
    h, w  = img.shape[:2]
    M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    img   = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    factor = random.uniform(0.75, 1.25)
    img    = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    return img.astype(np.float32) / 255.0

def save_img(img, folder, fname):
    img_uint8 = (img * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(folder, fname),
                cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR))

# Split 70% train / 15% val / 15% test — stratified
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.30, stratify=all_labels, random_state=42)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.50, stratify=temp_labels, random_state=42)

print(f"Train : {len(train_paths)} images")
print(f"  Anemic : {sum(train_labels)}")
print(f"  Normal : {train_labels.count(0)}")
print(f"Val   : {len(val_paths)} images")
print(f"  Anemic : {sum(val_labels)}")
print(f"  Normal : {val_labels.count(0)}")
print(f"Test  : {len(test_paths)} images")
print(f"  Anemic : {sum(test_labels)}")
print(f"  Normal : {test_labels.count(0)}")
print()

# Create folders
BASE_DIR = '/content/data'
for split in ['train', 'val', 'test']:
    for cls in ['anemic', 'normal']:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)

# Process train — with 2x augmentation
# 910 total → 637 train → ~1900 after augmentation
print("Processing train set with augmentation...")
failed = 0
for i, (path, label) in enumerate(zip(train_paths, train_labels)):
    cls  = 'anemic' if label == 1 else 'normal'
    folder = os.path.join(BASE_DIR, 'train', cls)
    img  = preprocess(path)
    if img is None:
        failed += 1
        continue
    save_img(img, folder, f"{i:04d}_orig.png")
    for j in range(2):
        save_img(augment(img), folder, f"{i:04d}_aug{j}.png")

# Process val — no augmentation
print("Processing val set...")
for i, (path, label) in enumerate(zip(val_paths, val_labels)):
    cls  = 'anemic' if label == 1 else 'normal'
    img  = preprocess(path)
    if img is None: continue
    save_img(img, os.path.join(BASE_DIR, 'val', cls), f"{i:04d}.png")

# Process test — no augmentation
print("Processing test set...")
for i, (path, label) in enumerate(zip(test_paths, test_labels)):
    cls  = 'anemic' if label == 1 else 'normal'
    img  = preprocess(path)
    if img is None: continue
    save_img(img, os.path.join(BASE_DIR, 'test', cls), f"{i:04d}.png")

print()
print("Final image counts:")
for split in ['train', 'val', 'test']:
    for cls in ['anemic', 'normal']:
        count = len(os.listdir(os.path.join(BASE_DIR, split, cls)))
        print(f"  {split}/{cls}: {count}")

if failed > 0:
    print(f"  Failed to load: {failed} images")

Train : 637 images
  Anemic : 349
  Normal : 288
Val   : 136 images
  Anemic : 75
  Normal : 61
Test  : 137 images
  Anemic : 75
  Normal : 62

Processing train set with augmentation...
Processing val set...
Processing test set...

Final image counts:
  train/anemic: 1047
  train/normal: 864
  val/anemic: 75
  val/normal: 61
  test/anemic: 75
  test/normal: 62


In [ ]:
# CELL 6 — Save split info + checklist
import json, os

DRIVE_PATH = '/content/drive/MyDrive/anemia_project'

# Save new split info
split_info = {
    'train_paths'  : train_paths,
    'val_paths'    : val_paths,
    'test_paths'   : test_paths,
    'train_labels' : train_labels,
    'val_labels'   : val_labels,
    'test_labels'  : test_labels,
    'base_dir'     : '/content/data'
}

split_path = os.path.join(DRIVE_PATH, 'split_info_combined.json')
with open(split_path, 'w') as f:
    json.dump(split_info, f, indent=2)
print(f"Saved: split_info_combined.json")

# Update config
config_path = os.path.join(DRIVE_PATH, 'config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

config['combined_labels_path'] = os.path.join(DRIVE_PATH, 'combined_labels.json')
config['split_info_combined']  = split_path
config['total_combined']       = 910
config['eyes_defy_count']      = 200
config['cp_anemic_count']      = 710
config['train_count']          = 1911
config['val_count']            = 136
config['test_count']           = 137

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print("Config updated!")
print()

# Checklist
BASE_DIR = '/content/data'
print("="*55)
print("  DAY 3 COMBINED — COMPLETE")
print("="*55)

checks = [
    ("Eyes Defy Anemia extracted",
     os.path.exists('/content/fresh_dataset/dataset anemia/Italy')),
    ("CP-AnemiC downloaded",
     os.path.exists('/content/cpanemic/extracted/Anemic')),
    ("Combined dict saved (910 images)",
     os.path.exists(os.path.join(DRIVE_PATH, 'combined_labels.json'))),
    ("Train folders ready",
     len(os.listdir(os.path.join(BASE_DIR,'train','anemic'))) > 0),
    ("Val folders ready",
     len(os.listdir(os.path.join(BASE_DIR,'val','anemic'))) > 0),
    ("Test folders ready",
     len(os.listdir(os.path.join(BASE_DIR,'test','anemic'))) > 0),
    ("split_info_combined.json saved",
     os.path.exists(split_path)),
    ("Config updated",
     'total_combined' in config),
]

all_ok = True
for name, result in checks:
    mark = "PASS" if result else "FAIL"
    print(f"  [{mark}]  {name}")
    if not result:
        all_ok = False

print()
print(f"  Eyes Defy Anemia : 200 images (Italy + India)")
print(f"  CP-AnemiC Ghana  : 710 images")
print(f"  Total combined   : 910 images — 3 countries")
print(f"  Train (3x aug)   : 1911 images")
print(f"  Val              : 136 images")
print(f"  Test             : 137 images")
print()
if all_ok:
    print("  ALL PASSED ")

else:
    print("  Some failed")
print("="*55)

Saved: split_info_combined.json
Config updated!

  DAY 3 COMBINED — COMPLETE
  [PASS]  Eyes Defy Anemia extracted
  [PASS]  CP-AnemiC downloaded
  [PASS]  Combined dict saved (910 images)
  [PASS]  Train folders ready
  [PASS]  Val folders ready
  [PASS]  Test folders ready
  [PASS]  split_info_combined.json saved
  [PASS]  Config updated

  Eyes Defy Anemia : 200 images (Italy + India)
  CP-AnemiC Ghana  : 710 images
  Total combined   : 910 images — 3 countries
  Train (3x aug)   : 1911 images
  Val              : 136 images
  Test             : 137 images

  ALL PASSED — Ready for Day 4 ML Baseline!
  Your dataset is now research grade.
  3 countries, 2 age groups, 910 patients.
